# 19 — Assembly101 Video-Only Knowledge Distillation Students

This notebook performs the main controlled student–teacher experiment.

## Fixed setup

- **Teacher:** the validation-selected `MS-TCN + CLIP` privileged teacher from notebook 18.
- **Teacher training/inference input:** CLIP video features plus the ground-truth action-text sequence.
- **Student input:** CLIP video features only.
- **Student validation, test, and deployment:** video only.
- **Controlled comparison:** CE-only student versus KD students.
- **KD ablation:** `lambda_KD ∈ {0.01, 0.05, 0.10}`, with temperature `T = 4`.
- **Model selection:** validation only.
- **Test:** evaluated only after the KD coefficient has been selected.

The teacher is an oracle because its text sequence is selected from ground-truth
labels. It is used only to provide soft targets during KD training. The final
student never receives text at inference.

## 1. Mount Google Drive

In [1]:
from google.colab import drive

drive.mount("/content/drive", force_remount=True)

Mounted at /content/drive


## 2. Imports and configuration

In [2]:
from pathlib import Path
from datetime import datetime, timezone

import copy
import gc
import hashlib
import json
import math
import random
import time
import traceback

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from tqdm.auto import tqdm

pd.set_option("display.max_columns", 180)
pd.set_option("display.max_colwidth", 240)

DRIVE_ROOT = Path("/content/drive/MyDrive/mmf_tas_lab_data")

FEATURE_ROOT = (
    DRIVE_ROOT
    / "text_assisted_tas"
    / "assembly101"
    / "coarse_mstcn_format"
    / "streaming_visual_features_v1"
)

DATA_ROOT = FEATURE_ROOT / "clip_vitb16"

TEACHER_ROOT = FEATURE_ROOT / "runs" / "18_text_assisted_concat_teachers"
TEACHER_PIPELINE_SUMMARY = TEACHER_ROOT / "final_summary_full.json"

VISUAL_BASELINE_SUMMARY = (
    FEATURE_ROOT
    / "runs"
    / "16_visual_only_mstcn"
    / "clip_vitb16"
    / "full"
    / "results"
    / "final_summary.json"
)

OUT_ROOT = FEATURE_ROOT / "runs" / "19_video_only_kd_students"
OUT_ROOT.mkdir(parents=True, exist_ok=True)

RUN_MODE = "full"
# RUN_MODE = "smoke"

SEED = 7

RUN_CONFIGS = {
    "smoke": {
        "epochs": 3,
        "max_train_sequences": 64,
        "max_val_sequences": 32,
        "max_test_sequences": 32,
        "eval_every": 1,
        "patience_evals": None,
        "kd_lambdas": [0.05],
    },
    "full": {
        "epochs": 120,
        "max_train_sequences": None,
        "max_val_sequences": None,
        "max_test_sequences": None,
        "eval_every": 5,
        "patience_evals": 12,
        "kd_lambdas": [0.01, 0.05, 0.10],
    },
}

if RUN_MODE not in RUN_CONFIGS:
    raise ValueError(f"Unknown RUN_MODE: {RUN_MODE}")

CFG = RUN_CONFIGS[RUN_MODE]

EXPECTED_VIDEO_DIM = 512
EXPECTED_TEXT_DIM = 512
EXPECTED_TEMPORAL_LENGTH = 16
EXPECTED_NUM_CLASSES = 202

NUM_STAGES = 4
NUM_LAYERS = 6
NUM_F_MAPS = 64
DROPOUT = 0.5

BATCH_SIZE = 64
NUM_WORKERS = 0

LEARNING_RATE = 5e-4
WEIGHT_DECAY = 1e-4
GRAD_CLIP = 5.0

SMOOTHING_WEIGHT = 0.15
SMOOTHING_CLIP_VALUE = 16.0

KD_TEMPERATURE = 4.0
KD_LAMBDAS = list(CFG["kd_lambdas"])
DISTILL_ALL_STAGES = True

RESUME_IF_AVAILABLE = True
FORCE_RETRAIN_COMPLETED = False
FORCE_RECREATE_SHARED_INITIALIZATION = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("RUN_MODE:", RUN_MODE)
print("CFG:", CFG)
print("KD_LAMBDAS:", KD_LAMBDAS)
print("DEVICE:", DEVICE)
print("OUT_ROOT:", OUT_ROOT)

if RUN_MODE == "full" and not torch.cuda.is_available():
    print("WARNING: full training is running without a GPU.")

PyTorch: 2.11.0+cpu
CUDA available: False
GPU: CPU
RUN_MODE: full
CFG: {'epochs': 120, 'max_train_sequences': None, 'max_val_sequences': None, 'max_test_sequences': None, 'eval_every': 5, 'patience_evals': 12, 'kd_lambdas': [0.01, 0.05, 0.1]}
KD_LAMBDAS: [0.01, 0.05, 0.1]
DEVICE: cpu
OUT_ROOT: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/streaming_visual_features_v1/runs/19_video_only_kd_students


## 3. Verify notebook 18 and resolve the selected teacher

In [3]:
for name, path in {
    "CLIP data": DATA_ROOT,
    "notebook 18 summary": TEACHER_PIPELINE_SUMMARY,
    "notebook 16 CLIP baseline": VISUAL_BASELINE_SUMMARY,
}.items():
    print(f"{name}: {path} -> {path.exists()}")
    if not path.exists():
        raise FileNotFoundError(f"Missing prerequisite: {name}: {path}")

teacher_pipeline = json.loads(
    TEACHER_PIPELINE_SUMMARY.read_text(encoding="utf-8")
)

if teacher_pipeline.get("status") != "completed":
    raise RuntimeError("Notebook 18 is not marked completed.")

recommended_teacher = teacher_pipeline["recommended_teacher_for_kd"]

if recommended_teacher["temporal_model"] != "mstcn":
    raise ValueError(
        "Notebook 19 expects the validation-selected teacher to be MS-TCN. "
        f"Found: {recommended_teacher['temporal_model']}"
    )

if recommended_teacher["representation"] != "clip_vitb16":
    raise ValueError(
        "Notebook 19 expects the validation-selected teacher to use CLIP video "
        f"features. Found: {recommended_teacher['representation']}"
    )

teacher_key = "mstcn/clip_vitb16"
teacher_experiment = teacher_pipeline["experiments"][teacher_key]

TEACHER_CHECKPOINT = Path(recommended_teacher["teacher_checkpoint"])
TRAIN_VIDEO_MEAN_PATH = Path(
    teacher_experiment["paths"]["train_video_mean"]
)
TRAIN_VIDEO_STD_PATH = Path(
    teacher_experiment["paths"]["train_video_std"]
)
TEXT_EMBEDDING_PATH = Path(
    teacher_experiment["input"]["text_embedding_path"]
)
TEXT_METADATA_PATH = Path(
    teacher_experiment["input"]["text_metadata_path"]
)

for name, path in {
    "teacher checkpoint": TEACHER_CHECKPOINT,
    "teacher train mean": TRAIN_VIDEO_MEAN_PATH,
    "teacher train std": TRAIN_VIDEO_STD_PATH,
    "text embeddings": TEXT_EMBEDDING_PATH,
    "text metadata": TEXT_METADATA_PATH,
}.items():
    print(f"{name}: {path} -> {path.exists()}")
    if not path.exists():
        raise FileNotFoundError(f"Missing teacher artifact: {name}: {path}")

visual_baseline_summary = json.loads(
    VISUAL_BASELINE_SUMMARY.read_text(encoding="utf-8")
)

if visual_baseline_summary.get("status") != "completed":
    raise RuntimeError("Notebook 16 CLIP baseline is not marked completed.")

print(
    json.dumps(
        {
            "selected_teacher": teacher_key,
            "selection_basis": recommended_teacher["selection_basis"],
            "validation_metrics": recommended_teacher["validation_metrics"],
            "oracle_test_metrics": teacher_experiment["test_metrics"],
            "checkpoint": str(TEACHER_CHECKPOINT),
            "deployable": False,
        },
        indent=2,
    )
)

CLIP data: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/streaming_visual_features_v1/clip_vitb16 -> True
notebook 18 summary: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/streaming_visual_features_v1/runs/18_text_assisted_concat_teachers/final_summary_full.json -> True
notebook 16 CLIP baseline: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/streaming_visual_features_v1/runs/16_visual_only_mstcn/clip_vitb16/full/results/final_summary.json -> True
teacher checkpoint: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/streaming_visual_features_v1/runs/18_text_assisted_concat_teachers/mstcn/clip_vitb16/full/models/best_teacher.pt -> True
teacher train mean: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/streaming_visual_features_v1/runs/18_text_assisted_concat_teachers/mstcn/cl

## 4. Reproducibility, mapping, and official split

In [4]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def read_nonempty_lines(path):
    return [
        line.strip()
        for line in Path(path).read_text(
            encoding="utf-8",
            errors="replace",
        ).splitlines()
        if line.strip()
    ]


def load_mapping(path):
    id_to_label = {}
    label_to_id = {}

    for line in read_nonempty_lines(path):
        class_id_text, label = line.split(maxsplit=1)
        class_id = int(class_id_text)

        if class_id in id_to_label:
            raise ValueError(f"Duplicate class ID: {class_id}")
        if label in label_to_id:
            raise ValueError(f"Duplicate label: {label}")

        id_to_label[class_id] = label
        label_to_id[label] = class_id

    return id_to_label, label_to_id


def resolve_bundle(split_dir, split_name):
    split_dir = Path(split_dir)

    for candidate in [
        split_dir / f"{split_name}.bundle",
        split_dir / f"{split_name}.split1.bundle",
    ]:
        if candidate.exists():
            return candidate

    candidates = sorted(
        path
        for path in split_dir.glob("*.bundle")
        if split_name.lower() in path.name.lower()
        and ".partial." not in path.name.lower()
        and not path.name.lower().endswith(".partial.bundle")
    )

    if len(candidates) != 1:
        raise FileNotFoundError(
            f"Could not uniquely resolve {split_name} in {split_dir}: {candidates}"
        )

    return candidates[0]


def read_bundle(path):
    sequence_ids = [Path(line).stem for line in read_nonempty_lines(path)]

    if len(sequence_ids) != len(set(sequence_ids)):
        raise ValueError(f"Duplicate IDs in {path}")

    return sequence_ids


def limit_ids(sequence_ids, maximum):
    if maximum is None:
        return list(sequence_ids)
    return list(sequence_ids)[: int(maximum)]


set_seed(SEED)

MAPPING_PATH = DATA_ROOT / "mapping.txt"
SPLIT_DIR = DATA_ROOT / "splits"

id_to_label, label_to_id = load_mapping(MAPPING_PATH)
num_classes = len(id_to_label)

if num_classes != EXPECTED_NUM_CLASSES:
    raise ValueError(
        f"Expected {EXPECTED_NUM_CLASSES} classes, found {num_classes}"
    )

if set(id_to_label) != set(range(num_classes)):
    raise ValueError("Class IDs must be contiguous from 0 to 201.")

train_ids_full = read_bundle(resolve_bundle(SPLIT_DIR, "train"))
val_ids_full = read_bundle(resolve_bundle(SPLIT_DIR, "val"))
test_ids_full = read_bundle(resolve_bundle(SPLIT_DIR, "test"))

assert set(train_ids_full).isdisjoint(set(val_ids_full))
assert set(train_ids_full).isdisjoint(set(test_ids_full))
assert set(val_ids_full).isdisjoint(set(test_ids_full))
assert len(set(train_ids_full) | set(val_ids_full) | set(test_ids_full)) == 680

train_ids = limit_ids(train_ids_full, CFG["max_train_sequences"])
val_ids = limit_ids(val_ids_full, CFG["max_val_sequences"])
test_ids = limit_ids(test_ids_full, CFG["max_test_sequences"])

print("Classes:", num_classes)
print("Train:", len(train_ids))
print("Validation:", len(val_ids))
print("Test:", len(test_ids))

Classes: 202
Train: 393
Validation: 120
Test: 167


## 5. Text bank, normalization, dataset, and loaders

In [5]:
text_embeddings = np.load(TEXT_EMBEDDING_PATH).astype(np.float32)
text_metadata = pd.read_csv(TEXT_METADATA_PATH)

required_columns = {"model_class_id", "action_cls", "prompt", "clip_model"}
missing_columns = required_columns - set(text_metadata.columns)

if missing_columns:
    raise KeyError(f"Missing text metadata columns: {sorted(missing_columns)}")

if text_embeddings.shape != (EXPECTED_NUM_CLASSES, EXPECTED_TEXT_DIM):
    raise ValueError(f"Unexpected text shape: {text_embeddings.shape}")

text_metadata = (
    text_metadata.sort_values("model_class_id").reset_index(drop=True)
)

if not np.array_equal(
    text_metadata["model_class_id"].to_numpy(),
    np.arange(EXPECTED_NUM_CLASSES),
):
    raise ValueError("Text metadata does not cover IDs 0..201 exactly.")

for class_id in range(num_classes):
    if str(text_metadata.loc[class_id, "action_cls"]) != str(id_to_label[class_id]):
        raise ValueError(
            f"Text/mapping mismatch at class {class_id}: "
            f"{text_metadata.loc[class_id, 'action_cls']} != {id_to_label[class_id]}"
        )

text_embeddings /= np.maximum(
    np.linalg.norm(text_embeddings, axis=1, keepdims=True),
    1e-12,
)
text_embeddings = text_embeddings.astype(np.float32)

train_video_mean = np.load(TRAIN_VIDEO_MEAN_PATH).astype(np.float32)
train_video_std = np.load(TRAIN_VIDEO_STD_PATH).astype(np.float32)

if train_video_mean.shape != (EXPECTED_VIDEO_DIM,):
    raise ValueError(f"Unexpected mean shape: {train_video_mean.shape}")
if train_video_std.shape != (EXPECTED_VIDEO_DIM,):
    raise ValueError(f"Unexpected std shape: {train_video_std.shape}")
if np.any(train_video_std <= 0):
    raise ValueError("All train standard deviations must be positive.")


class Assembly101KDDataset(Dataset):
    def __init__(self, sequence_ids):
        self.sequence_ids = list(sequence_ids)
        self.feature_dir = DATA_ROOT / "features"
        self.gt_dir = DATA_ROOT / "groundTruth"

    def __len__(self):
        return len(self.sequence_ids)

    def __getitem__(self, index):
        sequence_id = self.sequence_ids[index]

        video = np.load(
            self.feature_dir / f"{sequence_id}.npy"
        ).astype(np.float32)

        label_names = read_nonempty_lines(
            self.gt_dir / f"{sequence_id}.txt"
        )
        labels = np.asarray(
            [label_to_id[label] for label in label_names],
            dtype=np.int64,
        )

        if video.shape != (EXPECTED_VIDEO_DIM, EXPECTED_TEMPORAL_LENGTH):
            raise ValueError(
                f"Unexpected video shape for {sequence_id}: {video.shape}"
            )
        if labels.shape != (EXPECTED_TEMPORAL_LENGTH,):
            raise ValueError(
                f"Unexpected label shape for {sequence_id}: {labels.shape}"
            )
        if not np.isfinite(video).all():
            raise ValueError(f"Non-finite feature values for {sequence_id}")

        video = (
            video - train_video_mean[:, None]
        ) / (
            train_video_std[:, None] + 1e-8
        )

        privileged_text = text_embeddings[labels].T.astype(np.float32)

        return {
            "sequence_id": sequence_id,
            "video": torch.from_numpy(video),
            "privileged_text": torch.from_numpy(privileged_text),
            "labels": torch.from_numpy(labels),
        }


train_dataset = Assembly101KDDataset(train_ids)
val_dataset = Assembly101KDDataset(val_ids)
test_dataset = Assembly101KDDataset(test_ids)


def make_train_loader(epoch):
    generator = torch.Generator()
    generator.manual_seed(SEED + int(epoch))

    return DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
        generator=generator,
    )


val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

sample = next(iter(make_train_loader(1)))

assert sample["video"].shape[1:] == (
    EXPECTED_VIDEO_DIM,
    EXPECTED_TEMPORAL_LENGTH,
)
assert sample["privileged_text"].shape[1:] == (
    EXPECTED_TEXT_DIM,
    EXPECTED_TEMPORAL_LENGTH,
)
assert sample["labels"].shape[1:] == (EXPECTED_TEMPORAL_LENGTH,)

print("Text bank:", text_embeddings.shape)
print("Train/val/test:", len(train_dataset), len(val_dataset), len(test_dataset))
print("Video batch:", tuple(sample["video"].shape))
print("Privileged text batch:", tuple(sample["privileged_text"].shape))
print("Label batch:", tuple(sample["labels"].shape))

Text bank: (202, 512)
Train/val/test: 393 120 167
Video batch: (64, 512, 16)
Privileged text batch: (64, 512, 16)
Label batch: (64, 16)


## 6. Integrity scan

In [6]:
integrity_rows = []

active_train = set(train_ids)
active_val = set(val_ids)

for sequence_id in tqdm(
    list(train_ids) + list(val_ids) + list(test_ids),
    desc="Validate CLIP data",
):
    feature_path = DATA_ROOT / "features" / f"{sequence_id}.npy"
    gt_path = DATA_ROOT / "groundTruth" / f"{sequence_id}.txt"

    if not feature_path.exists():
        raise FileNotFoundError(feature_path)
    if not gt_path.exists():
        raise FileNotFoundError(gt_path)

    feature = np.load(feature_path, mmap_mode="r")
    labels = read_nonempty_lines(gt_path)
    unknown = sorted(set(labels) - set(label_to_id))

    if feature.shape != (EXPECTED_VIDEO_DIM, EXPECTED_TEMPORAL_LENGTH):
        raise ValueError(f"Unexpected shape for {sequence_id}: {feature.shape}")
    if len(labels) != EXPECTED_TEMPORAL_LENGTH:
        raise ValueError(f"Unexpected GT length for {sequence_id}: {len(labels)}")
    if not np.isfinite(feature).all():
        raise ValueError(f"Non-finite values in {feature_path}")
    if unknown:
        raise ValueError(f"Unknown labels in {gt_path}: {unknown}")

    split = (
        "train"
        if sequence_id in active_train
        else "val"
        if sequence_id in active_val
        else "test"
    )

    integrity_rows.append(
        {
            "sequence_id": sequence_id,
            "split": split,
            "feature_shape": str(tuple(feature.shape)),
            "gt_length": len(labels),
            "finite": True,
        }
    )

integrity_df = pd.DataFrame(integrity_rows)
integrity_path = OUT_ROOT / RUN_MODE / "dataset_integrity.csv"
integrity_path.parent.mkdir(parents=True, exist_ok=True)
integrity_df.to_csv(integrity_path, index=False)

display(
    integrity_df.groupby("split").agg(
        sequences=("sequence_id", "size"),
        feature_shapes=("feature_shape", "nunique"),
        gt_lengths=("gt_length", "nunique"),
        finite=("finite", "all"),
    ).reset_index()
)

print("Saved:", integrity_path)

Validate CLIP data:   0%|          | 0/680 [00:00<?, ?it/s]

,split,sequences,feature_shapes,gt_lengths,finite
0,test,167,1,1,True
1,train,393,1,1,True
2,val,120,1,1,True


Saved: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/streaming_visual_features_v1/runs/19_video_only_kd_students/full/dataset_integrity.csv


## 7. Temporal action segmentation metrics

In [7]:
IGNORE_CLASS_NAMES = {"background", "SIL", "silence"}
ignore_class_ids = {
    label_to_id[name]
    for name in IGNORE_CLASS_NAMES
    if name in label_to_id
}


def collapse_segments(frame_labels, ignored_ids=None):
    ignored_ids = set(ignored_ids or [])
    labels, starts, ends = [], [], []
    active = None

    for index, label in enumerate(frame_labels):
        label = int(label)

        if label in ignored_ids:
            if active is not None:
                ends.append(index)
                active = None
            continue

        if label != active:
            if active is not None:
                ends.append(index)
            labels.append(label)
            starts.append(index)
            active = label

    if active is not None:
        ends.append(len(frame_labels))

    return labels, starts, ends


def levenshtein_distance(predicted, target):
    distance = np.zeros(
        (len(predicted) + 1, len(target) + 1),
        dtype=np.int32,
    )
    distance[:, 0] = np.arange(len(predicted) + 1)
    distance[0, :] = np.arange(len(target) + 1)

    for row in range(1, len(predicted) + 1):
        for column in range(1, len(target) + 1):
            substitution = int(predicted[row - 1] != target[column - 1])
            distance[row, column] = min(
                distance[row - 1, column] + 1,
                distance[row, column - 1] + 1,
                distance[row - 1, column - 1] + substitution,
            )

    return int(distance[-1, -1])


def edit_score_single(prediction, target, ignored_ids=None):
    predicted_segments, _, _ = collapse_segments(prediction, ignored_ids)
    target_segments, _, _ = collapse_segments(target, ignored_ids)

    if not predicted_segments and not target_segments:
        return 100.0

    denominator = max(len(predicted_segments), len(target_segments))
    if denominator == 0:
        return 0.0

    distance = levenshtein_distance(predicted_segments, target_segments)
    return (1.0 - distance / denominator) * 100.0


def f_score_single(prediction, target, overlap, ignored_ids=None):
    predicted_labels, predicted_starts, predicted_ends = collapse_segments(
        prediction,
        ignored_ids,
    )
    target_labels, target_starts, target_ends = collapse_segments(
        target,
        ignored_ids,
    )

    true_positives = 0
    false_positives = 0
    hits = np.zeros(len(target_labels), dtype=np.float32)

    for predicted_index in range(len(predicted_labels)):
        best_iou = 0.0
        best_target = -1

        for target_index in range(len(target_labels)):
            if predicted_labels[predicted_index] != target_labels[target_index]:
                continue

            intersection = (
                min(predicted_ends[predicted_index], target_ends[target_index])
                - max(predicted_starts[predicted_index], target_starts[target_index])
            )
            union = (
                max(predicted_ends[predicted_index], target_ends[target_index])
                - min(predicted_starts[predicted_index], target_starts[target_index])
            )
            iou = max(intersection, 0) / union if union > 0 else 0.0

            if iou > best_iou:
                best_iou = iou
                best_target = target_index

        if (
            best_iou >= overlap
            and best_target >= 0
            and hits[best_target] == 0
        ):
            true_positives += 1
            hits[best_target] = 1
        else:
            false_positives += 1

    false_negatives = len(target_labels) - int(hits.sum())
    return true_positives, false_positives, false_negatives


def compute_metrics(predictions, targets, ignored_ids=None):
    total_correct = 0
    total_positions = 0
    edit_scores = []
    f_statistics = {
        0.10: [0, 0, 0],
        0.25: [0, 0, 0],
        0.50: [0, 0, 0],
    }

    for sequence_id, prediction in predictions.items():
        target = targets[sequence_id]

        total_correct += int((prediction == target).sum())
        total_positions += len(target)
        edit_scores.append(
            edit_score_single(prediction, target, ignored_ids)
        )

        for overlap in f_statistics:
            tp, fp, fn = f_score_single(
                prediction,
                target,
                overlap,
                ignored_ids,
            )
            f_statistics[overlap][0] += tp
            f_statistics[overlap][1] += fp
            f_statistics[overlap][2] += fn

    metrics = {
        "acc": 100.0 * total_correct / max(total_positions, 1),
        "edit": float(np.mean(edit_scores)) if edit_scores else 0.0,
    }

    for overlap, (tp, fp, fn) in f_statistics.items():
        precision = tp / max(tp + fp, 1e-8)
        recall = tp / max(tp + fn, 1e-8)
        f1 = 2.0 * precision * recall / max(precision + recall, 1e-8)
        metrics[f"f1@{int(overlap * 100)}"] = 100.0 * f1

    return metrics


def round_metrics(metrics):
    return {key: round(float(value), 2) for key, value in metrics.items()}


def selection_score(metrics):
    return float(metrics["f1@25"]) + 0.01 * float(metrics["edit"])


print("Ignored class IDs:", ignore_class_ids)
print("Metrics ready.")

Ignored class IDs: set()
Metrics ready.


## 8. Exact MS-TCN teacher and video-only student

In [8]:
class DilatedResidualLayer(nn.Module):
    def __init__(self, dilation, channels, dropout):
        super().__init__()
        self.conv_dilated = nn.Conv1d(
            channels,
            channels,
            kernel_size=3,
            padding=dilation,
            dilation=dilation,
        )
        self.conv_1x1 = nn.Conv1d(
            channels,
            channels,
            kernel_size=1,
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, inputs):
        output = F.relu(self.conv_dilated(inputs))
        output = self.conv_1x1(output)
        output = self.dropout(output)
        return inputs + output


class SingleStage(nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()

        self.input_projection = nn.Conv1d(
            input_dim,
            NUM_F_MAPS,
            kernel_size=1,
        )
        self.layers = nn.ModuleList(
            [
                DilatedResidualLayer(
                    dilation=2 ** layer_index,
                    channels=NUM_F_MAPS,
                    dropout=DROPOUT,
                )
                for layer_index in range(NUM_LAYERS)
            ]
        )
        self.output_projection = nn.Conv1d(
            NUM_F_MAPS,
            num_classes,
            kernel_size=1,
        )

    def forward(self, inputs):
        output = self.input_projection(inputs)

        for layer in self.layers:
            output = layer(output)

        return self.output_projection(output)


class MSTCNTeacher(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        self.stage1 = SingleStage(
            EXPECTED_VIDEO_DIM + EXPECTED_TEXT_DIM,
            num_classes,
        )
        self.refinement_stages = nn.ModuleList(
            [
                SingleStage(num_classes, num_classes)
                for _ in range(NUM_STAGES - 1)
            ]
        )

    def forward(self, video, privileged_text):
        logits = self.stage1(
            torch.cat([video, privileged_text], dim=1)
        )
        outputs = [logits]

        for stage in self.refinement_stages:
            logits = stage(F.softmax(logits, dim=1))
            outputs.append(logits)

        return torch.stack(outputs, dim=0)


class MSTCNVideoStudent(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        self.stage1 = SingleStage(EXPECTED_VIDEO_DIM, num_classes)
        self.refinement_stages = nn.ModuleList(
            [
                SingleStage(num_classes, num_classes)
                for _ in range(NUM_STAGES - 1)
            ]
        )

    def forward(self, video):
        logits = self.stage1(video)
        outputs = [logits]

        for stage in self.refinement_stages:
            logits = stage(F.softmax(logits, dim=1))
            outputs.append(logits)

        return torch.stack(outputs, dim=0)


def create_teacher():
    return MSTCNTeacher(num_classes).to(DEVICE)


def create_student():
    return MSTCNVideoStudent(num_classes).to(DEVICE)


teacher_shape_test = create_teacher()
student_shape_test = create_student()

with torch.no_grad():
    dummy_video = torch.zeros(
        2,
        EXPECTED_VIDEO_DIM,
        EXPECTED_TEMPORAL_LENGTH,
        device=DEVICE,
    )
    dummy_text = torch.zeros(
        2,
        EXPECTED_TEXT_DIM,
        EXPECTED_TEMPORAL_LENGTH,
        device=DEVICE,
    )

    teacher_output = teacher_shape_test(dummy_video, dummy_text)
    student_output = student_shape_test(dummy_video)

expected_shape = (
    NUM_STAGES,
    2,
    num_classes,
    EXPECTED_TEMPORAL_LENGTH,
)

assert teacher_output.shape == expected_shape
assert student_output.shape == expected_shape

print(
    "Teacher parameters:",
    sum(parameter.numel() for parameter in teacher_shape_test.parameters()),
)
print(
    "Student parameters:",
    sum(parameter.numel() for parameter in student_shape_test.parameters()),
)
print("Teacher output:", tuple(teacher_output.shape))
print("Student output:", tuple(student_output.shape))
print("Student inference signature: student(video)")

del (
    teacher_shape_test,
    student_shape_test,
    dummy_video,
    dummy_text,
    teacher_output,
    student_output,
)

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

Teacher parameters: 553384
Student parameters: 520616
Teacher output: (4, 2, 202, 16)
Student output: (4, 2, 202, 16)
Student inference signature: student(video)


## 9. Load and freeze the selected teacher

In [9]:
def load_checkpoint(path):
    try:
        return torch.load(
            path,
            map_location=DEVICE,
            weights_only=False,
        )
    except TypeError:
        return torch.load(path, map_location=DEVICE)


teacher_checkpoint = load_checkpoint(TEACHER_CHECKPOINT)

if teacher_checkpoint.get("temporal_model") != "mstcn":
    raise ValueError("Teacher checkpoint is not MS-TCN.")
if teacher_checkpoint.get("representation") != "clip_vitb16":
    raise ValueError("Teacher checkpoint does not use clip_vitb16.")

teacher = create_teacher()
load_result = teacher.load_state_dict(
    teacher_checkpoint["model_state_dict"],
    strict=True,
)
teacher.eval()

for parameter in teacher.parameters():
    parameter.requires_grad_(False)

if load_result.missing_keys or load_result.unexpected_keys:
    raise RuntimeError(
        f"Teacher load mismatch: missing={load_result.missing_keys}, "
        f"unexpected={load_result.unexpected_keys}"
    )

with torch.no_grad():
    smoke_teacher_output = teacher(
        sample["video"][:2].to(DEVICE),
        sample["privileged_text"][:2].to(DEVICE),
    )

assert smoke_teacher_output.shape == (
    NUM_STAGES,
    2,
    num_classes,
    EXPECTED_TEMPORAL_LENGTH,
)

print("Teacher checkpoint epoch:", teacher_checkpoint["epoch"])
print("Teacher output:", tuple(smoke_teacher_output.shape))
print(
    "Teacher trainable parameters:",
    sum(
        parameter.numel()
        for parameter in teacher.parameters()
        if parameter.requires_grad
    ),
)

del smoke_teacher_output
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

Teacher checkpoint epoch: 120
Teacher output: (4, 2, 202, 16)
Teacher trainable parameters: 0


## 10. Losses

In [10]:
def supervised_loss(outputs, targets):
    total = torch.zeros((), device=outputs.device)
    ce_sum = 0.0
    smooth_sum = 0.0

    for stage_index in range(outputs.shape[0]):
        logits = outputs[stage_index]

        ce = F.cross_entropy(
            logits.permute(0, 2, 1).reshape(-1, logits.shape[1]),
            targets.reshape(-1),
        )

        log_probabilities = F.log_softmax(logits, dim=1)
        smooth = F.mse_loss(
            log_probabilities[:, :, 1:],
            log_probabilities.detach()[:, :, :-1],
            reduction="none",
        )
        smooth = torch.clamp(
            smooth,
            min=0.0,
            max=SMOOTHING_CLIP_VALUE,
        ).mean()

        total = total + ce + SMOOTHING_WEIGHT * smooth
        ce_sum += float(ce.detach().cpu())
        smooth_sum += float(smooth.detach().cpu())

    return total, {
        "supervised": float(total.detach().cpu()),
        "cross_entropy_sum": ce_sum,
        "smoothing_sum": smooth_sum,
    }


def distillation_loss(student_outputs, teacher_outputs):
    if student_outputs.shape != teacher_outputs.shape:
        raise ValueError(
            f"Student/teacher output mismatch: "
            f"{student_outputs.shape} vs {teacher_outputs.shape}"
        )

    stage_indices = (
        range(student_outputs.shape[0])
        if DISTILL_ALL_STAGES
        else [student_outputs.shape[0] - 1]
    )

    losses = []

    for stage_index in stage_indices:
        student_logits = (
            student_outputs[stage_index]
            .permute(0, 2, 1)
            .reshape(-1, student_outputs.shape[2])
        )
        teacher_logits = (
            teacher_outputs[stage_index]
            .detach()
            .permute(0, 2, 1)
            .reshape(-1, teacher_outputs.shape[2])
        )

        student_log_probability = F.log_softmax(
            student_logits / KD_TEMPERATURE,
            dim=-1,
        )
        teacher_probability = F.softmax(
            teacher_logits / KD_TEMPERATURE,
            dim=-1,
        )

        stage_loss = F.kl_div(
            student_log_probability,
            teacher_probability,
            reduction="batchmean",
        ) * (KD_TEMPERATURE ** 2)

        losses.append(stage_loss)

    return torch.stack(losses).mean()


def total_student_loss(
    student_outputs,
    targets,
    kd_lambda,
    teacher_outputs=None,
):
    supervised, supervised_parts = supervised_loss(
        student_outputs,
        targets,
    )

    if kd_lambda > 0:
        if teacher_outputs is None:
            raise ValueError("teacher_outputs required for KD.")
        kd = distillation_loss(student_outputs, teacher_outputs)
    else:
        kd = torch.zeros((), device=student_outputs.device)

    total = supervised + float(kd_lambda) * kd

    return total, {
        **supervised_parts,
        "kd_unweighted": float(kd.detach().cpu()),
        "kd_weighted": float((float(kd_lambda) * kd).detach().cpu()),
        "total": float(total.detach().cpu()),
    }


print("Supervised loss:")
print("  sum_s [CE_s + 0.15 * temporal smoothing_s]")
print("KD loss:")
print("  mean_s [T^2 * KL(teacher_T || student_T)]")
print("Total:")
print("  L_supervised + lambda_KD * L_KD")
print("Temperature:", KD_TEMPERATURE)
print("Distill all stages:", DISTILL_ALL_STAGES)

Supervised loss:
  sum_s [CE_s + 0.15 * temporal smoothing_s]
KD loss:
  mean_s [T^2 * KL(teacher_T || student_T)]
Total:
  L_supervised + lambda_KD * L_KD
Temperature: 4.0
Distill all stages: True


## 11. Shared student initialization and video-only evaluation

In [11]:
SHARED_INIT_PATH = (
    OUT_ROOT
    / RUN_MODE
    / "shared_student_initialization.pt"
)
SHARED_INIT_PATH.parent.mkdir(parents=True, exist_ok=True)

shared_config = {
    "seed": SEED,
    "num_stages": NUM_STAGES,
    "num_layers": NUM_LAYERS,
    "num_f_maps": NUM_F_MAPS,
    "dropout": DROPOUT,
    "video_dim": EXPECTED_VIDEO_DIM,
    "num_classes": num_classes,
}


def fingerprint(state_dict):
    digest = hashlib.sha256()

    for key in sorted(state_dict):
        tensor = state_dict[key].detach().cpu().contiguous()
        digest.update(key.encode("utf-8"))
        digest.update(tensor.numpy().tobytes())

    return digest.hexdigest()


if SHARED_INIT_PATH.exists() and not FORCE_RECREATE_SHARED_INITIALIZATION:
    shared_payload = load_checkpoint(SHARED_INIT_PATH)

    if shared_payload.get("config") != shared_config:
        raise ValueError(
            "Existing shared initialization has incompatible configuration. "
            "Set FORCE_RECREATE_SHARED_INITIALIZATION=True."
        )

    shared_state = shared_payload["model_state_dict"]
else:
    set_seed(SEED)
    initial_model = create_student()

    shared_state = {
        key: value.detach().cpu().clone()
        for key, value in initial_model.state_dict().items()
    }

    torch.save(
        {
            "config": shared_config,
            "model_state_dict": shared_state,
            "fingerprint": fingerprint(shared_state),
        },
        SHARED_INIT_PATH,
    )

    del initial_model
    gc.collect()

shared_fingerprint = fingerprint(shared_state)


@torch.no_grad()
def evaluate_student(
    model,
    loader,
    save_predictions=False,
    prediction_dir=None,
):
    model.eval()
    predictions = {}
    targets = {}

    if save_predictions:
        prediction_dir = Path(prediction_dir)
        prediction_dir.mkdir(parents=True, exist_ok=True)

    for batch in loader:
        video = batch["video"].to(DEVICE, non_blocking=True)
        labels = batch["labels"].cpu().numpy()
        sequence_ids = list(batch["sequence_id"])

        # Deliberately video-only: privileged_text is not read here.
        outputs = model(video)
        predicted = outputs[-1].argmax(dim=1).detach().cpu().numpy()

        for index, sequence_id in enumerate(sequence_ids):
            pred_ids = predicted[index].astype(np.int64)
            target_ids = labels[index].astype(np.int64)

            predictions[sequence_id] = pred_ids
            targets[sequence_id] = target_ids

            if save_predictions:
                pred_labels = [id_to_label[int(value)] for value in pred_ids]
                (
                    prediction_dir / f"{sequence_id}.txt"
                ).write_text(
                    "\n".join(pred_labels) + "\n",
                    encoding="utf-8",
                )

    return (
        compute_metrics(predictions, targets, ignore_class_ids),
        predictions,
        targets,
    )


print("Shared initialization:", SHARED_INIT_PATH)
print("Fingerprint:", shared_fingerprint)
print("Validation/test inference: student(video)")

Shared initialization: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/streaming_visual_features_v1/runs/19_video_only_kd_students/full/shared_student_initialization.pt
Fingerprint: 9b6c82355bad7c451aa6ee11584877b3dee106a62bef930a1290ce8674159566
Validation/test inference: student(video)


## 12. Train one controlled student experiment

In [12]:
def lambda_name(value):
    return (
        f"{float(value):.4f}"
        .rstrip("0")
        .rstrip(".")
        .replace(".", "p")
    )


def experiment_name(kd_lambda):
    if float(kd_lambda) == 0.0:
        return "controlled_ce"
    return f"kd_lambda_{lambda_name(kd_lambda)}"


def train_experiment(kd_lambda):
    kd_lambda = float(kd_lambda)
    name = experiment_name(kd_lambda)

    experiment_root = OUT_ROOT / RUN_MODE / name
    model_dir = experiment_root / "models"
    result_dir = experiment_root / "results"

    model_dir.mkdir(parents=True, exist_ok=True)
    result_dir.mkdir(parents=True, exist_ok=True)

    best_path = model_dir / "best_student.pt"
    last_path = model_dir / "last_checkpoint.pt"
    history_path = result_dir / "training_history.csv"
    summary_path = result_dir / "validation_summary.json"

    if summary_path.exists() and not FORCE_RETRAIN_COMPLETED:
        previous = json.loads(summary_path.read_text(encoding="utf-8"))

        if previous.get("status") == "validation_complete":
            print("Skipping completed:", name)
            return previous

    set_seed(SEED)
    student = create_student()
    student.load_state_dict(shared_state, strict=True)

    current_fingerprint = fingerprint(
        {
            key: value.detach().cpu()
            for key, value in student.state_dict().items()
        }
    )
    if current_fingerprint != shared_fingerprint:
        raise RuntimeError("Shared initialization did not load exactly.")

    optimizer = torch.optim.AdamW(
        student.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=CFG["epochs"],
    )

    start_epoch = 1
    best_score = -math.inf
    best_epoch = None
    best_validation_metrics = None
    no_improvement_evals = 0
    history = []

    if RESUME_IF_AVAILABLE and last_path.exists():
        checkpoint = load_checkpoint(last_path)

        compatible = (
            checkpoint.get("run_mode") == RUN_MODE
            and float(checkpoint.get("kd_lambda")) == kd_lambda
            and checkpoint.get("initialization_fingerprint")
            == shared_fingerprint
        )

        if compatible:
            student.load_state_dict(checkpoint["model_state_dict"], strict=True)
            optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
            scheduler.load_state_dict(checkpoint["scheduler_state_dict"])

            start_epoch = int(checkpoint["epoch"]) + 1
            best_score = float(checkpoint.get("best_score", -math.inf))
            best_epoch = checkpoint.get("best_epoch")
            best_validation_metrics = checkpoint.get("best_validation_metrics")
            no_improvement_evals = int(
                checkpoint.get("no_improvement_evals", 0)
            )
            history = list(checkpoint.get("history", []))

            print("Resuming", name, "from epoch", start_epoch)

    print("\n" + "=" * 88)
    print("EXPERIMENT:", name)
    print("KD lambda:", kd_lambda)
    print("Teacher used during training:", kd_lambda > 0)
    print("Validation/test input: video only")
    print("=" * 88)

    start_time = time.time()
    stopped_early = False

    for epoch in range(start_epoch, CFG["epochs"] + 1):
        student.train()
        train_loader = make_train_loader(epoch)

        total_values = []
        supervised_values = []
        kd_values = []
        weighted_kd_values = []

        for batch in train_loader:
            video = batch["video"].to(DEVICE, non_blocking=True)
            labels = batch["labels"].to(DEVICE, non_blocking=True)

            if kd_lambda > 0:
                privileged_text = batch["privileged_text"].to(
                    DEVICE,
                    non_blocking=True,
                )

                with torch.no_grad():
                    teacher_outputs = teacher(video, privileged_text)
            else:
                teacher_outputs = None

            optimizer.zero_grad(set_to_none=True)

            student_outputs = student(video)
            loss, parts = total_student_loss(
                student_outputs,
                labels,
                kd_lambda,
                teacher_outputs,
            )

            loss.backward()
            torch.nn.utils.clip_grad_norm_(student.parameters(), GRAD_CLIP)
            optimizer.step()

            total_values.append(parts["total"])
            supervised_values.append(parts["supervised"])
            kd_values.append(parts["kd_unweighted"])
            weighted_kd_values.append(parts["kd_weighted"])

        scheduler.step()

        row = {
            "experiment": name,
            "kd_lambda": kd_lambda,
            "epoch": epoch,
            "train_total_loss": float(np.mean(total_values)),
            "train_supervised_loss": float(np.mean(supervised_values)),
            "train_kd_unweighted": float(np.mean(kd_values)),
            "train_kd_weighted": float(np.mean(weighted_kd_values)),
            "learning_rate": float(scheduler.get_last_lr()[0]),
        }

        evaluate_now = (
            epoch == 1
            or epoch % CFG["eval_every"] == 0
            or epoch == CFG["epochs"]
        )

        if evaluate_now:
            validation_metrics, _, _ = evaluate_student(
                student,
                val_loader,
            )
            row.update(
                {
                    f"val_{key}": value
                    for key, value in validation_metrics.items()
                }
            )

            score = selection_score(validation_metrics)
            improved = score > best_score

            if improved:
                best_score = score
                best_epoch = epoch
                best_validation_metrics = dict(validation_metrics)
                no_improvement_evals = 0

                torch.save(
                    {
                        "epoch": epoch,
                        "run_mode": RUN_MODE,
                        "experiment": name,
                        "kd_lambda": kd_lambda,
                        "kd_temperature": KD_TEMPERATURE,
                        "distill_all_stages": DISTILL_ALL_STAGES,
                        "student_input": "video only",
                        "student_deployable": True,
                        "teacher_checkpoint": (
                            str(TEACHER_CHECKPOINT)
                            if kd_lambda > 0
                            else None
                        ),
                        "initialization_fingerprint": shared_fingerprint,
                        "model_state_dict": student.state_dict(),
                        "validation_metrics": validation_metrics,
                        "selection_score": score,
                        "model_config": shared_config,
                    },
                    best_path,
                )
            else:
                no_improvement_evals += 1

            print(
                f"{name} epoch {epoch:03d} "
                f"total={row['train_total_loss']:.4f} "
                f"sup={row['train_supervised_loss']:.4f} "
                f"kd={row['train_kd_unweighted']:.4f} "
                f"val_acc={validation_metrics['acc']:.2f} "
                f"val_edit={validation_metrics['edit']:.2f} "
                f"val_f1@25={validation_metrics['f1@25']:.2f} "
                f"val_f1@50={validation_metrics['f1@50']:.2f} "
                f"best={'yes' if improved else 'no'}"
            )
        else:
            print(
                f"{name} epoch {epoch:03d} "
                f"total={row['train_total_loss']:.4f} "
                f"sup={row['train_supervised_loss']:.4f} "
                f"kd={row['train_kd_unweighted']:.4f}"
            )

        history.append(row)

        torch.save(
            {
                "epoch": epoch,
                "run_mode": RUN_MODE,
                "experiment": name,
                "kd_lambda": kd_lambda,
                "kd_temperature": KD_TEMPERATURE,
                "initialization_fingerprint": shared_fingerprint,
                "model_state_dict": student.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "scheduler_state_dict": scheduler.state_dict(),
                "best_score": best_score,
                "best_epoch": best_epoch,
                "best_validation_metrics": best_validation_metrics,
                "no_improvement_evals": no_improvement_evals,
                "history": history,
            },
            last_path,
        )
        pd.DataFrame(history).to_csv(history_path, index=False)

        patience = CFG["patience_evals"]

        if (
            evaluate_now
            and patience is not None
            and no_improvement_evals >= patience
        ):
            stopped_early = True
            print(
                "Early stopping after",
                no_improvement_evals,
                "validation evaluations without improvement.",
            )
            break

    if not best_path.exists():
        raise RuntimeError(f"No best checkpoint saved for {name}")

    best_checkpoint = load_checkpoint(best_path)

    summary = {
        "status": "validation_complete",
        "run_mode": RUN_MODE,
        "experiment": name,
        "kd_lambda": kd_lambda,
        "kd_temperature": KD_TEMPERATURE,
        "distill_all_stages": DISTILL_ALL_STAGES,
        "teacher_used_during_training": kd_lambda > 0,
        "teacher_checkpoint": (
            str(TEACHER_CHECKPOINT) if kd_lambda > 0 else None
        ),
        "student_training_input": (
            "video + labels + teacher soft targets"
            if kd_lambda > 0
            else "video + labels"
        ),
        "student_validation_input": "video only",
        "student_test_input": "video only",
        "student_deployable": True,
        "initialization_fingerprint": shared_fingerprint,
        "data": {
            "train_sequences": len(train_ids),
            "validation_sequences": len(val_ids),
            "test_sequences_reserved": len(test_ids),
            "video_shape": [
                EXPECTED_VIDEO_DIM,
                EXPECTED_TEMPORAL_LENGTH,
            ],
        },
        "training": {
            "maximum_epochs": CFG["epochs"],
            "best_epoch": int(best_checkpoint["epoch"]),
            "stopped_early": stopped_early,
            "minutes_this_run": (time.time() - start_time) / 60.0,
            "batch_size": BATCH_SIZE,
            "seed": SEED,
        },
        "best_validation_score": float(best_checkpoint["selection_score"]),
        "best_validation_metrics": round_metrics(
            best_checkpoint["validation_metrics"]
        ),
        "test_metrics": None,
        "paths": {
            "best_student": str(best_path),
            "last_checkpoint": str(last_path),
            "training_history": str(history_path),
        },
    }

    summary_path.write_text(
        json.dumps(summary, indent=2),
        encoding="utf-8",
    )

    print(
        json.dumps(
            {
                "experiment": name,
                "best_epoch": summary["training"]["best_epoch"],
                "best_validation_metrics": summary["best_validation_metrics"],
                "best_student": summary["paths"]["best_student"],
            },
            indent=2,
        )
    )

    del student, optimizer, scheduler
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return summary

## 13. Run CE-only and KD ablations

In [13]:
try:
    controlled_ce_summary = train_experiment(0.0)

    kd_summaries = []
    for kd_lambda in KD_LAMBDAS:
        kd_summaries.append(train_experiment(kd_lambda))

except Exception:
    error_dir = OUT_ROOT / RUN_MODE / "errors"
    error_dir.mkdir(parents=True, exist_ok=True)

    error_path = error_dir / "training_traceback.txt"
    error_path.write_text(traceback.format_exc(), encoding="utf-8")

    print("Saved traceback:", error_path)
    raise

print("Controlled CE:", controlled_ce_summary["status"])
print("KD runs:", len(kd_summaries))


EXPERIMENT: controlled_ce
KD lambda: 0.0
Teacher used during training: False
Validation/test input: video only
controlled_ce epoch 001 total=21.1382 sup=21.1382 kd=0.0000 val_acc=11.67 val_edit=3.88 val_f1@25=4.90 val_f1@50=0.61 best=yes
controlled_ce epoch 002 total=20.2542 sup=20.2542 kd=0.0000
controlled_ce epoch 003 total=18.9488 sup=18.9488 kd=0.0000
controlled_ce epoch 004 total=17.8740 sup=17.8740 kd=0.0000
controlled_ce epoch 005 total=17.1934 sup=17.1934 kd=0.0000 val_acc=10.57 val_edit=5.02 val_f1@25=4.05 val_f1@50=0.17 best=no
controlled_ce epoch 006 total=16.7227 sup=16.7227 kd=0.0000
controlled_ce epoch 007 total=16.5522 sup=16.5522 kd=0.0000
controlled_ce epoch 008 total=16.1939 sup=16.1939 kd=0.0000
controlled_ce epoch 009 total=16.0463 sup=16.0463 kd=0.0000
controlled_ce epoch 010 total=15.9537 sup=15.9537 kd=0.0000 val_acc=13.91 val_edit=5.87 val_f1@25=7.42 val_f1@50=0.51 best=yes
controlled_ce epoch 011 total=15.7266 sup=15.7266 kd=0.0000
controlled_ce epoch 012 tota

## 14. Select KD coefficient using validation only

In [14]:
validation_rows = []

for summary in [controlled_ce_summary, *kd_summaries]:
    metrics = summary["best_validation_metrics"]

    validation_rows.append(
        {
            "experiment": summary["experiment"],
            "kd_lambda": summary["kd_lambda"],
            "teacher_used": summary["teacher_used_during_training"],
            "best_epoch": summary["training"]["best_epoch"],
            "val_acc": metrics["acc"],
            "val_edit": metrics["edit"],
            "val_f1@10": metrics["f1@10"],
            "val_f1@25": metrics["f1@25"],
            "val_f1@50": metrics["f1@50"],
            "validation_selection_score": summary["best_validation_score"],
            "best_student": summary["paths"]["best_student"],
        }
    )

validation_df = (
    pd.DataFrame(validation_rows)
    .sort_values(
        ["validation_selection_score", "val_f1@25", "val_edit"],
        ascending=False,
    )
    .reset_index(drop=True)
)

kd_validation_df = (
    validation_df[validation_df["kd_lambda"] > 0]
    .copy()
    .sort_values(
        ["validation_selection_score", "val_f1@25", "val_edit"],
        ascending=False,
    )
    .reset_index(drop=True)
)

if kd_validation_df.empty:
    raise RuntimeError("No KD experiment completed.")

selected_kd_lambda = float(kd_validation_df.iloc[0]["kd_lambda"])
selected_kd_summary = next(
    summary
    for summary in kd_summaries
    if float(summary["kd_lambda"]) == selected_kd_lambda
)

validation_csv = OUT_ROOT / RUN_MODE / "validation_ablation.csv"
validation_md = OUT_ROOT / RUN_MODE / "validation_ablation.md"

validation_df.to_csv(validation_csv, index=False)
validation_md.write_text(
    validation_df.to_markdown(index=False),
    encoding="utf-8",
)

display(validation_df)

print("Selected KD lambda:", selected_kd_lambda)
print("Selection used test data:", False)
print("Saved:", validation_csv)

,experiment,kd_lambda,teacher_used,best_epoch,val_acc,val_edit,val_f1@10,val_f1@25,val_f1@50,validation_selection_score,best_student
0,kd_lambda_0p1,0.10,True,100,29.90,26.25,28.86,25.02,18.47,25.278568,/content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/streaming_visual_features_v1/runs/19_video_only_kd_students/full/kd_lambda_0p1/models/best_student.pt
1,kd_lambda_0p05,0.05,True,80,30.00,25.99,28.81,24.97,18.18,25.227876,/content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/streaming_visual_features_v1/runs/19_video_only_kd_students/full/kd_lambda_0p05/models/best_student.pt
2,controlled_ce,0.00,False,80,30.00,26.17,29.14,24.78,18.36,25.037081,/content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/streaming_visual_features_v1/runs/19_video_only_kd_students/full/controlled_ce/models/best_student.pt
3,kd_lambda_0p01,0.01,True,80,29.95,25.89,28.99,24.65,18.39,24.907668,/content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/streaming_visual_features_v1/runs/19_video_only_kd_students/full/kd_lambda_0p01/models/best_student.pt


Selected KD lambda: 0.1
Selection used test data: False
Saved: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/streaming_visual_features_v1/runs/19_video_only_kd_students/full/validation_ablation.csv


## 15. Final test: controlled CE and selected KD student only

In [15]:
def evaluate_checkpoint(summary, prediction_name):
    checkpoint_path = Path(summary["paths"]["best_student"])
    checkpoint = load_checkpoint(checkpoint_path)

    student = create_student()
    student.load_state_dict(
        checkpoint["model_state_dict"],
        strict=True,
    )
    student.eval()

    prediction_dir = (
        OUT_ROOT
        / RUN_MODE
        / "test_predictions"
        / prediction_name
    )

    metrics, _, _ = evaluate_student(
        student,
        test_loader,
        save_predictions=True,
        prediction_dir=prediction_dir,
    )

    prediction_count = len(list(prediction_dir.glob("*.txt")))

    if prediction_count != len(test_ids):
        raise RuntimeError(
            f"Expected {len(test_ids)} predictions, found {prediction_count}"
        )

    result = {
        "checkpoint": str(checkpoint_path),
        "best_epoch": int(checkpoint["epoch"]),
        "metrics": round_metrics(metrics),
        "prediction_dir": str(prediction_dir),
        "prediction_files": prediction_count,
        "inference_input": "video only",
        "teacher_called": False,
        "text_used": False,
    }

    del student
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return result


controlled_ce_test = evaluate_checkpoint(
    controlled_ce_summary,
    "controlled_ce",
)

selected_kd_test = evaluate_checkpoint(
    selected_kd_summary,
    f"selected_kd_lambda_{lambda_name(selected_kd_lambda)}",
)

print("Controlled CE test:")
print(json.dumps(controlled_ce_test, indent=2))

print("\nSelected KD test:")
print(json.dumps(selected_kd_test, indent=2))

Controlled CE test:
{
  "checkpoint": "/content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/streaming_visual_features_v1/runs/19_video_only_kd_students/full/controlled_ce/models/best_student.pt",
  "best_epoch": 80,
  "metrics": {
    "acc": 25.94,
    "edit": 24.75,
    "f1@10": 27.58,
    "f1@25": 23.14,
    "f1@50": 16.53
  },
  "prediction_dir": "/content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/streaming_visual_features_v1/runs/19_video_only_kd_students/full/test_predictions/controlled_ce",
  "prediction_files": 167,
  "inference_input": "video only",
  "teacher_called": false,
  "text_used": false
}

Selected KD test:
{
  "checkpoint": "/content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/streaming_visual_features_v1/runs/19_video_only_kd_students/full/kd_lambda_0p1/models/best_student.pt",
  "best_epoch": 100,
  "metrics": {
    "acc": 26.38,
    "edit": 24.64,
    "

## 16. Final comparison and summary

In [16]:
visual_metrics = visual_baseline_summary["test_metrics"]
teacher_oracle_metrics = teacher_experiment["test_metrics"]
ce_metrics = controlled_ce_test["metrics"]
kd_metrics = selected_kd_test["metrics"]

final_rows = [
    {
        "experiment": "notebook16_visual_baseline",
        "model": "MS-TCN visual-only",
        "training_text": False,
        "inference_text": False,
        "deployable": True,
        "kd_lambda": None,
        "best_epoch": visual_baseline_summary["training"]["best_epoch"],
        **{key: float(visual_metrics[key]) for key in [
            "acc", "edit", "f1@10", "f1@25", "f1@50"
        ]},
        "note": "External visual-only reference from notebook 16.",
    },
    {
        "experiment": "controlled_ce_student",
        "model": "MS-TCN video-only student",
        "training_text": False,
        "inference_text": False,
        "deployable": True,
        "kd_lambda": 0.0,
        "best_epoch": controlled_ce_test["best_epoch"],
        **ce_metrics,
        "note": "Primary same-notebook CE-only control.",
    },
    {
        "experiment": "privileged_concat_teacher",
        "model": "MS-TCN video + GT-text oracle teacher",
        "training_text": True,
        "inference_text": True,
        "deployable": False,
        "kd_lambda": None,
        "best_epoch": teacher_experiment["training"]["best_epoch"],
        **{key: float(teacher_oracle_metrics[key]) for key in [
            "acc", "edit", "f1@10", "f1@25", "f1@50"
        ]},
        "note": "Oracle upper bound; requires ground-truth text at test.",
    },
    {
        "experiment": "selected_kd_student",
        "model": "MS-TCN video-only KD student",
        "training_text": True,
        "inference_text": False,
        "deployable": True,
        "kd_lambda": selected_kd_lambda,
        "best_epoch": selected_kd_test["best_epoch"],
        **kd_metrics,
        "note": "KD coefficient selected using validation only.",
    },
]

final_df = pd.DataFrame(final_rows)
final_csv = OUT_ROOT / RUN_MODE / "final_test_comparison.csv"
final_md = OUT_ROOT / RUN_MODE / "final_test_comparison.md"

final_df.to_csv(final_csv, index=False)
final_md.write_text(final_df.to_markdown(index=False), encoding="utf-8")

delta_rows = []

for metric in ["acc", "edit", "f1@10", "f1@25", "f1@50"]:
    delta_rows.append(
        {
            "metric": metric,
            "kd_minus_controlled_ce": round(
                float(kd_metrics[metric]) - float(ce_metrics[metric]),
                2,
            ),
            "kd_minus_notebook16_visual": round(
                float(kd_metrics[metric]) - float(visual_metrics[metric]),
                2,
            ),
            "teacher_minus_kd": round(
                float(teacher_oracle_metrics[metric]) - float(kd_metrics[metric]),
                2,
            ),
        }
    )

delta_df = pd.DataFrame(delta_rows)
delta_csv = OUT_ROOT / RUN_MODE / "metric_deltas.csv"
delta_df.to_csv(delta_csv, index=False)

display(final_df)
display(delta_df)

loss_definition = {
    "supervised": (
        "sum over stages of [cross entropy + "
        "0.15 * clipped temporal log-probability MSE]"
    ),
    "distillation": (
        "mean over selected stages of "
        "T^2 * KL(teacher_softmax(logits/T) || "
        "student_softmax(logits/T)); batch and temporal positions flattened"
    ),
    "total": "L_supervised + lambda_KD * L_KD",
    "temperature": KD_TEMPERATURE,
    "distill_all_stages": DISTILL_ALL_STAGES,
}

final_summary = {
    "status": "completed",
    "run_mode": RUN_MODE,
    "experiment": "assembly101_clip_mstcn_video_only_student_kd",
    "data": {
        "representation": "clip_vitb16",
        "video_shape": [EXPECTED_VIDEO_DIM, EXPECTED_TEMPORAL_LENGTH],
        "num_classes": num_classes,
        "train_sequences": len(train_ids),
        "validation_sequences": len(val_ids),
        "test_sequences": len(test_ids),
    },
    "teacher": {
        "architecture": "MS-TCN",
        "checkpoint": str(TEACHER_CHECKPOINT),
        "training_input": "video + ground-truth action text",
        "inference_input": "video + ground-truth action text",
        "deployable": False,
        "oracle_test_metrics": teacher_oracle_metrics,
    },
    "student": {
        "architecture": "MS-TCN",
        "input": "video only",
        "inference_text": False,
        "deployable": True,
        "shared_initialization": shared_fingerprint,
    },
    "losses": loss_definition,
    "selection_protocol": {
        "checkpoint_selection": "validation F1@25 + 0.01 * validation Edit",
        "kd_lambda_selection": "validation only",
        "test_access": (
            "controlled CE and selected KD evaluated after validation selection"
        ),
    },
    "kd_ablation": validation_df.to_dict(orient="records"),
    "selected_kd_lambda": selected_kd_lambda,
    "controlled_ce": {
        "validation": controlled_ce_summary["best_validation_metrics"],
        "test": controlled_ce_test,
    },
    "selected_kd_student": {
        "validation": selected_kd_summary["best_validation_metrics"],
        "test": selected_kd_test,
    },
    "external_visual_baseline": {
        "source": "notebook 16",
        "test_metrics": visual_metrics,
    },
    "metric_deltas": delta_df.to_dict(orient="records"),
    "paths": {
        "output_root": str(OUT_ROOT),
        "validation_ablation": str(validation_csv),
        "final_test_comparison": str(final_csv),
        "metric_deltas": str(delta_csv),
        "controlled_ce_predictions": controlled_ce_test["prediction_dir"],
        "selected_kd_predictions": selected_kd_test["prediction_dir"],
    },
    "interpretation_constraints": [
        "The privileged teacher is an oracle and is not deployable.",
        (
            "The primary controlled comparison is selected KD student versus "
            "the same-notebook controlled CE student."
        ),
        (
            "The student uses video only during validation, test, and deployment."
        ),
    ],
    "next_step": (
        "Notebook 20: aggregate Breakfast and Assembly101 experiments, "
        "generate final ablation tables, plots, and presentation-ready results."
    ),
    "created_utc": datetime.now(timezone.utc).isoformat(),
}

final_summary_path = OUT_ROOT / RUN_MODE / "final_summary.json"
final_summary_path.write_text(
    json.dumps(final_summary, indent=2),
    encoding="utf-8",
)

print("Saved:", validation_csv)
print("Saved:", final_csv)
print("Saved:", delta_csv)
print("Saved:", final_summary_path)

print(
    json.dumps(
        {
            "status": final_summary["status"],
            "selected_kd_lambda": selected_kd_lambda,
            "controlled_ce_test": ce_metrics,
            "selected_kd_test": kd_metrics,
            "final_summary": str(final_summary_path),
        },
        indent=2,
    )
)

print("\nNext notebook:")
print(
    "20_assembly101_final_ablation_analysis_and_presentation_artifacts_COLAB.ipynb"
)

,experiment,model,training_text,inference_text,deployable,kd_lambda,best_epoch,acc,edit,f1@10,f1@25,f1@50,note
0,notebook16_visual_baseline,MS-TCN visual-only,False,False,True,NaN,110,25.86,25.01,28.00,23.82,17.58,External visual-only reference from notebook 16.
1,controlled_ce_student,MS-TCN video-only student,False,False,True,0.0,80,25.94,24.75,27.58,23.14,16.53,Primary same-notebook CE-only control.
2,privileged_concat_teacher,MS-TCN video + GT-text oracle teacher,True,True,False,NaN,120,44.27,37.39,41.65,40.94,36.81,Oracle upper bound; requires ground-truth text at test.
3,selected_kd_student,MS-TCN video-only KD student,True,False,True,0.1,100,26.38,24.64,28.40,23.75,17.73,KD coefficient selected using validation only.


,metric,kd_minus_controlled_ce,kd_minus_notebook16_visual,teacher_minus_kd
0,acc,0.44,0.52,17.89
1,edit,-0.11,-0.37,12.75
2,f1@10,0.82,0.40,13.25
3,f1@25,0.61,-0.07,17.19
4,f1@50,1.20,0.15,19.08


Saved: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/streaming_visual_features_v1/runs/19_video_only_kd_students/full/validation_ablation.csv
Saved: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/streaming_visual_features_v1/runs/19_video_only_kd_students/full/final_test_comparison.csv
Saved: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/streaming_visual_features_v1/runs/19_video_only_kd_students/full/metric_deltas.csv
Saved: /content/drive/MyDrive/mmf_tas_lab_data/text_assisted_tas/assembly101/coarse_mstcn_format/streaming_visual_features_v1/runs/19_video_only_kd_students/full/final_summary.json
{
  "status": "completed",
  "selected_kd_lambda": 0.1,
  "controlled_ce_test": {
    "acc": 25.94,
    "edit": 24.75,
    "f1@10": 27.58,
    "f1@25": 23.14,
    "f1@50": 16.53
  },
  "selected_kd_test": {
    "acc": 26.38,
    "edit": 24.64,
    "f1@10": 28.4,
  

## Completion criterion

A complete run writes:

```text
.../runs/19_video_only_kd_students/full/
├── shared_student_initialization.pt
├── controlled_ce/
├── kd_lambda_0p01/
├── kd_lambda_0p05/
├── kd_lambda_0p1/
├── test_predictions/
├── validation_ablation.csv
├── final_test_comparison.csv
├── metric_deltas.csv
└── final_summary.json
```

The main result is the difference between the selected **video-only KD student**
and the same-notebook **video-only CE control**.